# Predição de λmax — CCM-109 (UFABC)
Notebook inicial do projeto: predição do comprimento de onda de máxima absorção (λmax, nm) de moléculas orgânicas a partir do SMILES.

**Etapa 1:** treino/comparação de modelos sobre o Deep4Chem.
**Etapa 2:** aplicação do melhor modelo no conjunto proprietário (5.974 moléculas, sem rótulo).

Modelos incluídos, já com as sugestões do professor:
- Baseline 1: MLP + fingerprints Morgan
- Baseline 2 (forte): Gradient Boosting sobre descritores RDKit
- Modelo principal: Bi-LSTM char-level sobre SMILES
- Extensão (estado da arte): Chemprop (D-MPNN) — célula separada, opcional
- SMILES do solvente como feature adicional em todos os modelos
- Scaffold split (Bemis-Murcko) em vez de split aleatório
- Métricas: MAE e RMSE (nm)
- Etapa 2 com incerteza via ensemble/MC-Dropout


In [ ]:
# Instalação de dependências (rodar no Google Colab)
!pip install -q rdkit torch scikit-learn lightgbm pandas numpy matplotlib pyarrow
# Chemprop é opcional e pesado — só instale se for usar a célula de GNN no final
# !pip install -q chemprop


In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors
from rdkit.Chem.Scaffolds import MurckoScaffold

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


## 1. Dados — Deep4Chem
Baixe o dataset a partir do Figshare (DOI: 10.6084/m9.figshare.12045567.v2) e ajuste o nome do arquivo
conforme o que for extraído (geralmente um .csv ou .xlsx dentro do zip do Figshare).


In [ ]:
# Baixar o Deep4Chem (rodar no Colab, que tem acesso à internet liberado)
!wget -q -O deep4chem.zip "https://ndownloader.figshare.com/articles/12045567/versions/2"
!unzip -o -q deep4chem.zip -d deep4chem_raw
!ls deep4chem_raw
# Ajuste o caminho abaixo conforme o nome real do arquivo extraído
RAW_PATH = "/content/deep4chem_raw/DB for chromophore_Sci_Data_rev02.csv"  # <-- ajustar
df_raw = pd.read_csv(RAW_PATH)
df_raw.head()


In [ ]:
# Padronizar nomes de colunas (ajustar conforme o cabeçalho real do Deep4Chem)
# Colunas esperadas: SMILES do cromóforo, SMILES do solvente, lambda_max de absorção
COL_MAP = {
    "Chromophore": "smiles",
    "Solvent": "solvent_smiles",
    "Absorption max (nm)": "lambda_max",
}
df = df_raw.rename(columns=COL_MAP)[list(COL_MAP.values())].dropna()

def valid_smiles(s):
    return Chem.MolFromSmiles(s) is not None

df = df[df["smiles"].apply(valid_smiles) & df["solvent_smiles"].apply(valid_smiles)]
df = df.drop_duplicates(subset=["smiles", "solvent_smiles"]).reset_index(drop=True)
print(len(df), "registros válidos")
df.head()


## 2. Scaffold split
Split por molécula (Bemis-Murcko scaffold), não aleatório, para estimar generalização real —
evita que a mesma "família" de molécula apareça em treino e teste.


In [ ]:
def get_scaffold(smiles):
    mol = Chem.MolFromSmiles(smiles)
    try:
        scaffold = MurckoScaffold.MurckoScaffoldSmiles(mol=mol, includeChirality=False)
    except Exception:
        scaffold = smiles
    return scaffold

def scaffold_split(df, test_size=0.2, seed=SEED):
    df = df.copy()
    df["scaffold"] = df["smiles"].apply(get_scaffold)
    scaffolds = df["scaffold"].unique()
    rng = np.random.RandomState(seed)
    rng.shuffle(scaffolds)
    n_test = int(len(scaffolds) * test_size)
    test_scaffolds = set(scaffolds[:n_test])
    test_mask = df["scaffold"].isin(test_scaffolds)
    return df[~test_mask].reset_index(drop=True), df[test_mask].reset_index(drop=True)

train_df, test_df = scaffold_split(df, test_size=0.2)
print("train:", len(train_df), "test:", len(test_df))


## 3. Features
Fingerprints Morgan (chromóforo + solvente) e descritores RDKit. O SMILES do solvente entra
como feature adicional em todos os modelos, já que λmax depende do solvente.


In [ ]:
from rdkit.Chem import rdFingerprintGenerator

_morgan_gens = {}

def morgan_fp(smiles, n_bits=2048, radius=2):
    mol = Chem.MolFromSmiles(smiles)
    key = (radius, n_bits)
    if key not in _morgan_gens:
        _morgan_gens[key] = rdFingerprintGenerator.GetMorganGenerator(radius=radius, fpSize=n_bits)
    return _morgan_gens[key].GetFingerprintAsNumPy(mol).astype(np.float32)

def rdkit_descriptors(smiles):
    mol = Chem.MolFromSmiles(smiles)
    return np.array([f(mol) for _, f in Descriptors.descList], dtype=np.float32)

def build_fp_features(frame):
    X_mol = np.stack(frame["smiles"].apply(morgan_fp).values)
    X_solv = np.stack(frame["solvent_smiles"].apply(lambda s: morgan_fp(s, n_bits=256)).values)
    return np.concatenate([X_mol, X_solv], axis=1)

def build_descriptor_features(frame):
    X_mol = np.stack(frame["smiles"].apply(rdkit_descriptors).values)
    X_solv = np.stack(frame["solvent_smiles"].apply(rdkit_descriptors).values)
    X = np.concatenate([X_mol, X_solv], axis=1)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
    return X

y_train = train_df["lambda_max"].values.astype(np.float32)
y_test = test_df["lambda_max"].values.astype(np.float32)


## 4. Baseline 1 — MLP + fingerprints Morgan


In [7]:
class MLP(nn.Module):
    def __init__(self, in_dim, hidden=512, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden, hidden // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden // 2, 1),
        )
    def forward(self, x):
        return self.net(x).squeeze(-1)

def train_torch_model(model, X_train, y_train, X_val, y_val, epochs=100, lr=1e-3, batch_size=64):
    model = model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.L1Loss()  # MAE
    X_train_t = torch.tensor(X_train, dtype=torch.float32)
    y_train_t = torch.tensor(y_train, dtype=torch.float32)
    ds = torch.utils.data.TensorDataset(X_train_t, y_train_t)
    dl = DataLoader(ds, batch_size=batch_size, shuffle=True)
    X_val_t = torch.tensor(X_val, dtype=torch.float32).to(device)
    for epoch in range(epochs):
        model.train()
        for xb, yb in dl:
            xb, yb = xb.to(device), yb.to(device)
            opt.zero_grad()
            pred = model(xb)
            loss = loss_fn(pred, yb)
            loss.backward()
            opt.step()
        if (epoch + 1) % 20 == 0:
            model.eval()
            with torch.no_grad():
                val_pred = model(X_val_t).cpu().numpy()
            mae = mean_absolute_error(y_val, val_pred)
            print(f"epoch {epoch+1} — val MAE: {mae:.2f} nm")
    return model

X_train_fp = build_fp_features(train_df)
X_test_fp = build_fp_features(test_df)

mlp_model = MLP(in_dim=X_train_fp.shape[1])
mlp_model = train_torch_model(mlp_model, X_train_fp, y_train, X_test_fp, y_test)


epoch 20 — val MAE: 36.90 nm
epoch 40 — val MAE: 35.04 nm
epoch 60 — val MAE: 33.91 nm
epoch 80 — val MAE: 33.80 nm
epoch 100 — val MAE: 34.35 nm


## 5. Baseline 2 (forte) — Gradient Boosting sobre descritores RDKit


## 6. Modelo principal — Bi-LSTM char-level sobre SMILES
O solvente entra concatenado como fingerprint Morgan (256 bits) junto à saída da LSTM antes das camadas densas finais.


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

X_train_desc = build_descriptor_features(train_df)
X_test_desc = build_descriptor_features(test_df)

gbm = GradientBoostingRegressor(
    n_estimators=500,
    max_depth=4,
    learning_rate=0.05,
    random_state=SEED
)

gbm.fit(X_train_desc, y_train)

gbm_pred = gbm.predict(X_test_desc)

print(
    "GBM MAE:",
    mean_absolute_error(y_test, gbm_pred),
    "RMSE:",
    root_mean_squared_error(y_test, gbm_pred)
)

In [ ]:
# Vocabulário de caracteres a partir do próprio conjunto de treino
all_chars = sorted(set("".join(train_df["smiles"].tolist())))
char2idx = {c: i + 1 for i, c in enumerate(all_chars)}  # 0 = padding
vocab_size = len(char2idx) + 1
MAX_LEN = int(train_df["smiles"].str.len().quantile(0.99)) + 2

def encode_smiles(s, max_len=MAX_LEN):
    idxs = [char2idx.get(c, 0) for c in s[:max_len]]
    idxs = idxs + [0] * (max_len - len(idxs))
    return np.array(idxs, dtype=np.int64)

class SmilesDataset(Dataset):
    def __init__(self, frame, y):
        self.smiles_enc = np.stack(frame["smiles"].apply(encode_smiles).values)
        self.solv_fp = np.stack(frame["solvent_smiles"].apply(lambda s: morgan_fp(s, n_bits=256)).values)
        self.y = y
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.smiles_enc[idx], self.solv_fp[idx], self.y[idx]

class BiLSTMRegressor(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, hidden=128, solv_dim=256, dropout=0.2):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.lstm = nn.LSTM(embed_dim, hidden, bidirectional=True, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden * 2 + solv_dim, 256), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(256, 1),
        )
    def forward(self, smiles_idx, solv_fp):
        emb = self.embed(smiles_idx)
        _, (h_n, _) = self.lstm(emb)
        h_cat = torch.cat([h_n[-2], h_n[-1]], dim=-1)  # última camada, ambas direções
        x = torch.cat([h_cat, solv_fp], dim=-1)
        return self.head(x).squeeze(-1)

train_ds = SmilesDataset(train_df, y_train)
test_ds = SmilesDataset(test_df, y_test)
train_dl = DataLoader(train_ds, batch_size=64, shuffle=True)
test_dl = DataLoader(test_ds, batch_size=256, shuffle=False)

bilstm = BiLSTMRegressor(vocab_size).to(device)
opt = torch.optim.Adam(bilstm.parameters(), lr=1e-3)
loss_fn = nn.L1Loss()

for epoch in range(60):
    bilstm.train()
    for smiles_idx, solv_fp, yb in train_dl:
        smiles_idx, solv_fp, yb = smiles_idx.to(device), solv_fp.to(device), yb.to(device)
        opt.zero_grad()
        pred = bilstm(smiles_idx, solv_fp)
        loss = loss_fn(pred, yb)
        loss.backward()
        opt.step()
    if (epoch + 1) % 10 == 0:
        bilstm.eval()
        preds, trues = [], []
        with torch.no_grad():
            for smiles_idx, solv_fp, yb in test_dl:
                smiles_idx, solv_fp = smiles_idx.to(device), solv_fp.to(device)
                preds.append(bilstm(smiles_idx, solv_fp).cpu().numpy())
                trues.append(yb.numpy())
        preds, trues = np.concatenate(preds), np.concatenate(trues)
        print(f"epoch {epoch+1} — MAE: {mean_absolute_error(trues, preds):.2f} nm")


## 7. (Opcional) Extensão estado da arte — Chemprop (D-MPNN)
Requer `pip install chemprop` e os dados em formato CSV com colunas de SMILES + target.
Deixe comentado até decidir incluir essa comparação na entrega final.


In [ ]:
# train_df[["smiles", "lambda_max"]].to_csv("chemprop_train.csv", index=False)
# test_df[["smiles", "lambda_max"]].to_csv("chemprop_test.csv", index=False)
# !chemprop_train --data_path chemprop_train.csv --separate_test_path chemprop_test.csv \
#     --dataset_type regression --target_columns lambda_max --smiles_columns smiles \
#     --metric mae --extra_metrics rmse --save_dir chemprop_model


## 8. Comparação de resultados (MAE e RMSE, nm)


In [ ]:
bilstm.eval()
preds, trues = [], []
with torch.no_grad():
    for smiles_idx, solv_fp, yb in test_dl:
        smiles_idx, solv_fp = smiles_idx.to(device), solv_fp.to(device)
        preds.append(bilstm(smiles_idx, solv_fp).cpu().numpy())
        trues.append(yb.numpy())
bilstm_pred = np.concatenate(preds)

mlp_model.eval()
with torch.no_grad():
    mlp_pred = mlp_model(torch.tensor(X_test_fp, dtype=torch.float32).to(device)).cpu().numpy()

def report(name, y_true, y_pred):
    return {
        "modelo": name,
        "MAE (nm)": mean_absolute_error(y_true, y_pred),
        "RMSE (nm)": mean_squared_error(y_true, y_pred, squared=False),
    }

results = pd.DataFrame([
    report("MLP + Morgan (baseline)", y_test, mlp_pred),
    report("Gradient Boosting + descritores", y_test, gbm_pred),
    report("Bi-LSTM (SMILES + solvente)", y_test, bilstm_pred),
])
results


## 9. Etapa 2 — Aplicação no conjunto proprietário (5.974 moléculas) + incerteza
Como o ranking guiará síntese experimental, priorize candidatos com **alta confiança**.
Abaixo, um esqueleto de MC-Dropout (mantém dropout ativo na inferência e roda várias passadas).


In [ ]:
# Carregamento do conjunto proprietário — ajustado ao formato real do arquivo
# (colunas de proveniência dos anéis/fragmentos + coluna final 'smiles')
PROP_PATH = "banco_smiles_aromatizados.parquet"
prop_raw = pd.read_parquet(PROP_PATH)

# Mantém as colunas de ID (rastreiam de quais anéis/fragmentos cada molécula veio),
# úteis depois para interpretar o ranking e planejar a síntese
ID_COLS = [c for c in prop_raw.columns if c.lower().startswith("id_")]
SMILES_COL = "smiles"  # coluna final, equivalente a SMILES_Aromatico

prop_df = prop_raw[ID_COLS + [SMILES_COL]].copy()

def try_parse(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is not None:
        return mol
    # Alguns SMILES do gerador usam ':' explícito entre átomos maiúsculos (aromaticidade
    # explícita) em vez de átomos minúsculos — tenta parsear sem sanitizar e sanitiza à parte
    mol = Chem.MolFromSmiles(smiles, sanitize=False)
    if mol is None:
        return None
    try:
        Chem.SanitizeMol(mol)
        return mol
    except Exception:
        return None

prop_df["mol_ok"] = prop_df["smiles"].apply(lambda s: try_parse(s) is not None)
n_ok, n_total = prop_df["mol_ok"].sum(), len(prop_df)
print(f"{n_ok} / {n_total} SMILES válidos ({n_ok/n_total:.1%})")
if n_ok < n_total:
    print("Exemplos que falharam na sanitização:")
    display(prop_df[~prop_df["mol_ok"]].head(5))

prop_df = prop_df[prop_df["mol_ok"]].drop(columns="mol_ok").reset_index(drop=True)


In [ ]:
# Nota: o conjunto proprietário não tem solvente rotulado — decidir um solvente de referência
# (ex.: o mais frequente no Deep4Chem, ou o de interesse do laboratório) e usar como constante.
REFERENCE_SOLVENT = "CCO"  # <-- ajustar (ex.: etanol) conforme o contexto experimental do ABC Sim

def mc_dropout_predict(model, smiles_series, solvent_smiles, n_passes=30):
    model.train()  # mantém dropout ativo de propósito
    smiles_enc = np.stack(smiles_series.apply(encode_smiles).values)
    solv_fp = np.tile(morgan_fp(solvent_smiles, n_bits=256), (len(smiles_series), 1))
    smiles_t = torch.tensor(smiles_enc, dtype=torch.int64).to(device)
    solv_t = torch.tensor(solv_fp, dtype=torch.float32).to(device)
    all_preds = []
    with torch.no_grad():
        for _ in range(n_passes):
            all_preds.append(model(smiles_t, solv_t).cpu().numpy())
    all_preds = np.stack(all_preds)
    return all_preds.mean(axis=0), all_preds.std(axis=0)

mean_pred, std_pred = mc_dropout_predict(bilstm, prop_df["smiles"], REFERENCE_SOLVENT)
prop_df["lambda_max_pred"] = mean_pred
prop_df["incerteza_std"] = std_pred

# Mantém as colunas de ID no ranking final para rastrear a origem de cada candidato
ranking = prop_df.sort_values(["incerteza_std", "lambda_max_pred"], ascending=[True, False])
ranking.to_csv("ranking_lambda_max_proprietario.csv", index=False)
ranking.head(20)


## 10. Próximos passos sugeridos
- Ajustar `RAW_PATH`, `COL_MAP` e `PROP_PATH` conforme os arquivos reais.
- Avaliar deslocamento de distribuição: comparar estatísticas de descritores (peso molecular, logP, etc.) entre Deep4Chem e o conjunto proprietário.
- Rodar validação cruzada (K-fold sobre scaffolds) para seleção de hiperparâmetros, não só um único scaffold split.
- Se houver tempo, adicionar Chemprop (D-MPNN) como comparação de estado da arte.
